# 📊 Livrable 1 — DATA : Préparation des données & Dashboard

**Problématique :** *Dans quelle mesure l'IA peut-elle servir de levier de régulation face à l'engorgement touristique parisien, en réorientant les flux de visiteurs vers la richesse culturelle de la petite et grande couronne ?*

---

## Ce que fait ce notebook

| Étape | Ce qu'on fait | Pourquoi |
|-------|--------------|----------|
| 1 | Extraction des données brutes | Les fichiers Excel sont des "rapports", pas des bases de données. On va chercher chaque chiffre par son nom. |
| 2 | Nettoyage & Feature Engineering | On crée de nouvelles variables utiles (part de Paris, zone géographique, croissance...) |
| 3 | Export des CSV propres | Pour pouvoir les réutiliser dans le Livrable 2 (Machine Learning) |
| 4 | Dashboard (6 graphiques) | Chaque graphique = 1 argument de notre problématique |


## 🔧 Étape 0 — Installation et imports

On installe les librairies nécessaires et on configure l'environnement.


In [ ]:
# Installation (décommenter si besoin)
# !pip install openpyxl matplotlib seaborn pandas

import openpyxl
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Style des graphiques
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3
})

print("✅ Imports OK")


## ⚙️ Configuration

**⚠️ IMPORTANT : Modifier `DATA_DIR` ci-dessous** pour pointer vers le dossier où tu as mis les fichiers Excel.

Par exemple :
- Google Colab : `"/content/drive/MyDrive/hackathon/"`
- Databricks : `"/dbfs/FileStore/hackathon/"`
- Jupyter local : `"./data/"`


In [ ]:
# ══════════════════════════════════════════════════════════════
# ⬇️  MODIFIER CETTE LIGNE pour pointer vers ton dossier data  ⬇️
# ══════════════════════════════════════════════════════════════
DATA_DIR = "./data/"   # ← Mettre le chemin vers tes fichiers Excel
# ══════════════════════════════════════════════════════════════

YEARS = list(range(2014, 2026))  # 2014 à 2025

# Les 8 départements d'Île-de-France (dans l'ordre des colonnes Excel)
DEPTS = [
    "Paris",
    "Seine-et-Marne",
    "Yvelines",
    "Essonne",
    "Hauts-de-Seine",
    "Seine-Saint-Denis",
    "Val-de-Marne",
    "Val-d'Oise"
]

# Couleurs pour les graphiques
COLORS = {
    "Paris": "#E63946",
    "Seine-et-Marne": "#457B9D",
    "Yvelines": "#2A9D8F",
    "Essonne": "#E9C46A",
    "Hauts-de-Seine": "#F4A261",
    "Seine-Saint-Denis": "#264653",
    "Val-de-Marne": "#A8DADC",
    "Val-d'Oise": "#6A994E"
}

# Vérifier que les fichiers existent
missing = []
for y in YEARS:
    for prefix in ["ATR_Departements_Franciliens_", "ATR_Pays_"]:
        f = os.path.join(DATA_DIR, f"{prefix}{y}.xlsx")
        if not os.path.exists(f):
            missing.append(f)

if missing:
    print(f"❌ {len(missing)} fichiers manquants ! Vérifie DATA_DIR.")
    for m in missing[:5]:
        print(f"   → {m}")
    if len(missing) > 5:
        print(f"   ... et {len(missing)-5} autres")
else:
    print(f"✅ Tous les {len(YEARS)*2} fichiers Excel trouvés dans {DATA_DIR}")


## 📥 Étape 1 — Fonctions d'extraction

### Pourquoi ces fonctions ?

Les fichiers Excel ATR ne sont **pas** des bases de données classiques. Ce sont des **rapports visuels** :
- Plein de lignes vides, de titres de section, d'astérisques
- Les numéros de ligne **changent d'une année à l'autre** (ex: "Satisfaction propreté" est à la ligne 810 en 2014 mais à la ligne 838 en 2025)

**Notre solution** : on cherche chaque indicateur **par son nom** (pas par son numéro de ligne). C'est plus lent mais beaucoup plus fiable.


In [ ]:
def load_excel_as_rows(filepath):
    """
    Charge un fichier Excel ATR et retourne une LISTE ORDONNÉE de tuples :
      [ (row_index, "nom_indicateur", [val_total, val_paris, ...]), ... ]

    Pourquoi une liste ordonnée (et pas un dictionnaire) ?
      → Certains indicateurs ont le MÊME NOM dans différentes sections
        (ex: "La propreté" apparaît 3 fois). L'ordre nous permet
        de savoir dans quel BLOC on se trouve.
    """
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    ws = wb['Résultats']

    rows = []
    for i, row in enumerate(ws.iter_rows(values_only=True)):
        label = row[0]
        if label and isinstance(label, str):
            clean_label = label.strip().lstrip('\u00dc').strip()
            if clean_label:
                values = list(row[1:])
                rows.append((i, clean_label, values))
    wb.close()
    return rows


def extract_value(rows_list, label_search, col_index=0):
    """
    Cherche un indicateur par son nom et retourne la valeur demandée.

    col_index : 0=Total IDF, 1=Paris, 2=Seine-et-Marne, ...
    Retourne None si pas trouvé ou si c'est un texte / astérisque.
    """
    for _, label, values in rows_list:
        if label_search.lower() in label.lower():
            if col_index < len(values):
                val = values[col_index]
                if val is None or val == '*' or isinstance(val, str):
                    return None
                return float(val)
    return None


def extract_satisfaction_recap(rows_list, indicator_name, col_index=0):
    """
    Cas spécial pour la satisfaction.

    Problème : "La propreté" apparaît 3 fois dans le fichier :
      1. Comme titre de section (ligne ~773)
      2. Dans le bloc "% satisfait" (c'est celui qu'on veut → valeur entre 0 et 1)
      3. Dans le bloc "Indices de satisfaction" (valeur entre 0 et 100)

    Solution : on cherche d'abord le header "% satisfait",
    puis on lit l'indicateur DANS ce bloc uniquement.
    """
    in_recap_block = False
    for _, label, values in rows_list:
        if '% satisfait' in label and 'très' in label.lower():
            in_recap_block = True
            continue
        if in_recap_block and 'Indices de satisfaction' in label:
            break
        if in_recap_block and indicator_name.lower() in label.lower():
            if col_index < len(values):
                val = values[col_index]
                if val is None or val == '*' or isinstance(val, str):
                    return None
                return float(val)
            return None
    return None

print("✅ Fonctions d'extraction prêtes")


## 📂 Étape 1A — Extraction des données Départements

On lit les **12 fichiers** `ATR_Departements_Franciliens_20XX.xlsx` et on extrait pour chaque année × chaque département :

| Catégorie | Indicateurs extraits |
|-----------|---------------------|
| **Volume** | Séjours, Nuitées, Consommation touristique |
| **Comportement** | Durée de séjour, Repeaters, Hébergement, Motif |
| **Économie** | Dépense/jour/personne, Budget du séjour |
| **Perception** | Satisfaction globale, NPS, Intention de revenir |
| **Transport** | Part avion, train, route |
| **Satisfaction détaillée** | Propreté, sécurité, restauration, culture, transport |


In [ ]:
dept_records = []

for year in YEARS:
    filepath = os.path.join(DATA_DIR, f"ATR_Departements_Franciliens_{year}.xlsx")
    print(f"  📖 {year}...", end=" ")

    data = load_excel_as_rows(filepath)

    for dept_idx, dept_name in enumerate(DEPTS):
        col = dept_idx + 1  # col 0 = Total IDF, col 1 = Paris, etc.

        record = {
            "annee": year,
            "departement": dept_name,

            # Volume
            "sejours": extract_value(data, "Nombre de séjours", col),
            "nuitees": extract_value(data, "Nombre de nuitées", col),
            "conso_touristique": extract_value(data, "Consommation touristique", col),

            # Comportement
            "duree_sejour": extract_value(data, "Durée du séjour (en nb de nuits)", col),
            "repeaters": extract_value(data, "Repeaters", col),
            "hebergement_marchand": extract_value(data, "Hébergement marchand", col),
            "voyagent_individuel": extract_value(data, "Voyagent en individuel", col),
            "motif_personnel": extract_value(data, "Voyagent pour motifs personnels", col),

            # Économie
            "depense_jour_personne": extract_value(data, "Dépense moyenne par jour et par personne", col),
            "budget_sejour": extract_value(data, "Budget du séjour par personne", col),

            # Perception
            "satisfaction_globale": extract_value(data, "Satisfaction globale du séjour", col),
            "nps": extract_value(data, "Recommandation de la destination (NPS)", col),
            "intention_revenir": extract_value(data, "Souhaitent revenir d'ici 1 à 2 ans", col),

            # Transport
            "transport_avion": extract_value(data, "Part des touristes venus en avion", col),
            "transport_train": extract_value(data, "Part des touristes venus en train", col),
            "transport_route": extract_value(data, "Part des touristes venus par la route", col),
        }

        # Satisfaction détaillée
        for sat_label, sat_key in [
            ("propreté", "satisfaction_proprete"),
            ("sécurité", "satisfaction_securite"),
            ("Restauration", "satisfaction_restauration"),
            ("Transport", "satisfaction_transport"),
            ("culturelle", "satisfaction_culture"),
        ]:
            record[sat_key] = extract_satisfaction_recap(data, sat_label, col)

        # Total IDF pour calculer les parts
        record["sejours_total_idf"] = extract_value(data, "Nombre de séjours", 0)
        record["nuitees_total_idf"] = extract_value(data, "Nombre de nuitées", 0)

        dept_records.append(record)

    print("✓")

df_dept = pd.DataFrame(dept_records)
print(f"\n✅ {len(df_dept)} lignes extraites ({len(YEARS)} ans × {len(DEPTS)} départements)")
df_dept.head()


## 📂 Étape 1B — Extraction des données Pays

On lit les **12 fichiers** `ATR_Pays_20XX.xlsx` pour avoir le profil de chaque marché émetteur (nationalité).

Ces données serviront à identifier **quels touristes sont les plus faciles à réorienter** vers la couronne.


In [ ]:
pays_records = []

for year in YEARS:
    filepath = os.path.join(DATA_DIR, f"ATR_Pays_{year}.xlsx")
    print(f"  📖 {year}...", end=" ")

    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    ws = wb['Résultats']

    headers = []
    rows_data = {}
    for i, row in enumerate(ws.iter_rows(values_only=True)):
        if i == 6:
            headers = [str(h).replace(f'{year}_', '').strip() if h else '' for h in row]
        label = str(row[0]).strip().lstrip('\u00dc').strip() if row[0] else ''
        if label:
            rows_data[label] = list(row[1:])
    wb.close()

    country_names = headers[1:]

    for col_idx, country in enumerate(country_names):
        if not country or country.strip() == '':
            continue

        record = {
            "annee": year,
            "pays": country,
            "sejours": None, "nuitees": None, "duree_sejour": None,
            "repeaters": None, "depense_jour_personne": None,
            "satisfaction_globale": None, "nps": None,
            "transport_avion": None, "transport_train": None,
            "transport_route": None, "motif_personnel": None,
        }

        for label, values in rows_data.items():
            if col_idx < len(values):
                val = values[col_idx]
                if val is None or val == '*' or isinstance(val, str):
                    val = None
                else:
                    val = float(val)

                if 'Nombre de séjours' in label: record["sejours"] = val
                elif 'Nombre de nuitées' in label: record["nuitees"] = val
                elif 'Durée du séjour' in label and 'recodée' not in label: record["duree_sejour"] = val
                elif label == 'Repeaters': record["repeaters"] = val
                elif 'Dépense moyenne par jour' in label: record["depense_jour_personne"] = val
                elif 'Satisfaction globale' in label: record["satisfaction_globale"] = val
                elif 'Recommandation de la destination' in label: record["nps"] = val
                elif 'venus en avion' in label: record["transport_avion"] = val
                elif 'venus en train' in label: record["transport_train"] = val
                elif 'venus par la route' in label: record["transport_route"] = val
                elif 'motifs personnels uniquement' in label: record["motif_personnel"] = val

        pays_records.append(record)
    print("✓")

df_pays = pd.DataFrame(pays_records)
print(f"\n✅ {len(df_pays)} lignes extraites")
df_pays.head()


## 🧹 Étape 2 — Nettoyage & Feature Engineering

### Pourquoi cette étape ?

Les données brutes ont des problèmes :
- **2020-2021 = COVID** → 17M de séjours au lieu de 50M, ça fausse les tendances
- Les taux sont en décimal (0.63) au lieu de pourcentages (63%) → moins lisible
- Il nous manque des **variables calculées** pour raconter notre histoire

### Les 4 nouvelles variables qu'on crée :

| Variable | Calcul | Pourquoi c'est utile |
|----------|--------|---------------------|
| `part_sejours_idf` | séjours du dept ÷ séjours total IDF | **LE** chiffre clé : montre que Paris = 60% |
| `zone` | Paris / Petite couronne / Grande couronne | Simplifie la lecture (3 zones au lieu de 8 depts) |
| `evol_sejours_pct` | variation d'une année à l'autre | Montre que la couronne croît à +11% |
| `depense_totale_estimee` | dépense/jour × durée × séjours | Le potentiel économique de chaque département |


In [ ]:
# Marquer les années COVID (on ne les exclut pas, on les signale)
df_dept["is_covid"] = df_dept["annee"].isin([2020, 2021])
df_pays["is_covid"] = df_pays["annee"].isin([2020, 2021])

# Convertir les taux (0-1) en pourcentages (0-100)
pct_cols = ["repeaters", "satisfaction_globale", "hebergement_marchand",
            "voyagent_individuel", "motif_personnel",
            "transport_avion", "transport_train", "transport_route",
            "satisfaction_proprete", "satisfaction_securite",
            "satisfaction_restauration", "satisfaction_transport",
            "satisfaction_culture", "intention_revenir"]

for col in pct_cols:
    if col in df_dept.columns:
        df_dept[f"{col}_pct"] = df_dept[col] * 100

# VARIABLE 1 : Part de chaque département dans le total IDF
df_dept["part_sejours_idf"] = (df_dept["sejours"] / df_dept["sejours_total_idf"]) * 100

# VARIABLE 2 : Zone géographique
def zone(dept):
    if dept == "Paris": return "Paris"
    elif dept in ["Hauts-de-Seine", "Seine-Saint-Denis", "Val-de-Marne"]: return "Petite couronne"
    else: return "Grande couronne"

df_dept["zone"] = df_dept["departement"].apply(zone)

# VARIABLE 3 : Évolution annuelle des séjours
df_dept = df_dept.sort_values(["departement", "annee"])
df_dept["evol_sejours_pct"] = df_dept.groupby("departement")["sejours"].pct_change() * 100

# VARIABLE 4 : Potentiel économique
df_dept["depense_totale_estimee"] = (
    df_dept["depense_jour_personne"] * df_dept["duree_sejour"] * df_dept["sejours"]
)

print(f"✅ Données départements : {len(df_dept)} lignes, {len(df_dept.columns)} colonnes")
print(f"   Valeurs manquantes : {df_dept.isnull().sum().sum()} sur {df_dept.size}")
print(f"\n📋 Aperçu des données nettoyées :")
df_dept[["annee","departement","zone","sejours","part_sejours_idf",
         "depense_jour_personne","satisfaction_globale_pct","nps"]].tail(8)


In [ ]:
# Tableau simplifié : Paris vs Petite couronne vs Grande couronne
df_zone = df_dept.groupby(["annee", "zone", "is_covid"]).agg(
    sejours=("sejours", "sum"),
    nuitees=("nuitees", "sum"),
    conso_touristique=("conso_touristique", "sum"),
    sejours_total_idf=("sejours_total_idf", "first"),
    depense_jour_moy=("depense_jour_personne", "mean"),
    satisfaction_moy=("satisfaction_globale", "mean"),
    nps_moy=("nps", "mean"),
    satisfaction_proprete_moy=("satisfaction_proprete", "mean"),
    satisfaction_securite_moy=("satisfaction_securite", "mean"),
).reset_index()

df_zone["part_sejours_idf"] = (df_zone["sejours"] / df_zone["sejours_total_idf"]) * 100

print(f"✅ Tableau par zone : {len(df_zone)} lignes")
df_zone.head(6)


## 💾 Étape 3 — Export des données nettoyées

On sauvegarde **3 fichiers CSV** qui serviront de base au **Livrable 2 (Machine Learning)**.


In [ ]:
df_dept.to_csv("data_departements_clean.csv", index=False)
df_zone.to_csv("data_zones_clean.csv", index=False)
df_pays.to_csv("data_pays_clean.csv", index=False)

print("✅ Fichiers exportés :")
print(f"   📄 data_departements_clean.csv  ({len(df_dept)} lignes × {len(df_dept.columns)} colonnes)")
print(f"   📄 data_zones_clean.csv         ({len(df_zone)} lignes)")
print(f"   📄 data_pays_clean.csv          ({len(df_pays)} lignes)")


---

## 📊 Étape 4 — Dashboard

**6 graphiques** qui racontent l'histoire de notre problématique, dans l'ordre :

1. 🔴 **Le constat** → Paris concentre ~60% des séjours
2. 📊 **Le volume** → Vue d'ensemble des séjours par zone
3. ⭐ **La qualité** → La couronne est mieux notée que Paris
4. 📈 **La dynamique** → La couronne explose, Paris recule
5. 🎯 **La cible** → Quels marchés sont les plus réorientables
6. 💰 **Le potentiel** → Combien ça rapporterait


### 📊 Graphique 1 — Paris concentre ~60% des séjours (et ça s'aggrave)

**Argument pour le jury** : La part de Paris dans les séjours IDF passe de 54.9% en 2014 à 61.2% en 2024. Le problème de concentration s'aggrave structurellement.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

zone_colors = {"Paris": "#E63946", "Petite couronne": "#457B9D", "Grande couronne": "#2A9D8F"}

for z in ["Paris", "Petite couronne", "Grande couronne"]:
    sub = df_zone[df_zone["zone"] == z].sort_values("annee")
    ax.plot(sub["annee"], sub["part_sejours_idf"], marker='o', linewidth=2.5,
            label=z, color=zone_colors[z], markersize=6)
    last = sub.iloc[-1]
    ax.annotate(f'{last["part_sejours_idf"]:.1f}%',
                xy=(last["annee"], last["part_sejours_idf"]),
                xytext=(8, 0), textcoords='offset points', fontsize=10,
                fontweight='bold', color=zone_colors[z])

ax.axvspan(2019.5, 2021.5, alpha=0.1, color='gray', label='COVID')
ax.set_xlabel("Année"); ax.set_ylabel("Part des séjours IDF (%)")
ax.set_title("Paris concentre ~60% des séjours en Île-de-France",
             fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=11, loc='center right')
ax.set_xticks(YEARS); ax.set_xticklabels(YEARS, rotation=45)
ax.set_ylim(0, 70)
plt.tight_layout()
plt.savefig("graph1_concentration_paris.png", dpi=150)
plt.show()


### 📊 Graphique 2 — Volume de séjours par zone (2014-2025)

**Argument** : En valeur absolue, l'IDF a retrouvé son niveau pré-COVID (~49M de séjours). La structure de répartition entre zones est visible.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

pivot = df_zone.pivot_table(index="annee", columns="zone", values="sejours") / 1e6
pivot = pivot[["Grande couronne", "Petite couronne", "Paris"]]

pivot.plot(kind='bar', stacked=True, ax=ax,
           color=["#2A9D8F", "#457B9D", "#E63946"], width=0.7)

ax.set_ylabel("Séjours (millions)"); ax.set_xlabel("")
ax.set_title("Volume de séjours par zone (2014-2025)",
             fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=11, loc='upper left')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f M'))

for i, year in enumerate(pivot.index):
    total = pivot.loc[year].sum()
    ax.text(i, total + 0.5, f'{total:.0f}M', ha='center', fontsize=8, fontweight='bold')

plt.xticks(rotation=45); plt.tight_layout()
plt.savefig("graph2_volume_sejours_zones.png", dpi=150)
plt.show()


### 📊 Graphique 3 — Satisfaction comparée par département (2025)

**Argument clé** : Sur les critères sensibles (propreté, sécurité), la couronne fait **mieux** que Paris.
- Propreté : Seine-et-Marne 75.5% vs Paris 63.8%
- Sécurité : Seine-et-Marne 78.5% vs Paris 72.4%

→ Le surtourisme dégrade l'expérience à Paris, pas en couronne.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df_last = df_dept[df_dept["annee"] == 2025].copy()

sat_metrics = [
    ("satisfaction_globale_pct", "Satisfaction globale", axes[0]),
    ("satisfaction_proprete_pct", "Satisfaction propreté", axes[1]),
    ("satisfaction_securite_pct", "Satisfaction sécurité", axes[2]),
]

for col, title, ax in sat_metrics:
    vals = df_last[["departement", col]].dropna().sort_values(col, ascending=True)
    colors = [COLORS.get(d, "#999") for d in vals["departement"]]
    bars = ax.barh(vals["departement"], vals[col], color=colors)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlim(40, 105)
    for bar, val in zip(bars, vals[col]):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9)

fig.suptitle("Satisfaction par département (2025)", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("graph3_satisfaction_comparee.png", dpi=150, bbox_inches='tight')
plt.show()


### 📊 Graphique 4 — Croissance 2024→2025 : Paris recule, la couronne avance

**Argument** : La tendance est déjà en marche. Seine-et-Marne +11%, Yvelines +11%, Val-de-Marne +12%, Essonne +12%. Paris fait -1.7%. L'IA peut **accélérer** cette redistribution naturelle.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

df_evol = df_dept[df_dept["annee"] == 2025][["departement", "evol_sejours_pct"]].dropna()
df_evol = df_evol.sort_values("evol_sejours_pct", ascending=True)

colors = ["#E63946" if d == "Paris" else "#2A9D8F" for d in df_evol["departement"]]
bars = ax.barh(df_evol["departement"], df_evol["evol_sejours_pct"], color=colors)

for bar, val in zip(bars, df_evol["evol_sejours_pct"]):
    x_pos = bar.get_width() + 0.3 if val >= 0 else bar.get_width() - 0.3
    ha = 'left' if val >= 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'{val:+.1f}%', va='center', ha=ha, fontsize=11, fontweight='bold')

ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel("Évolution des séjours (%)")
ax.set_title("Croissance 2024→2025 : Paris recule, la couronne avance",
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig("graph4_croissance_departements.png", dpi=150)
plt.show()


### 📊 Graphique 5 — Quels marchés sont les plus réorientables ?

**Argument** : On croise 2 critères pour identifier les cibles :
- **Taux de Repeaters** (axe X) : un touriste qui revient a déjà vu Paris → il est ouvert à la couronne
- **Part venus par la route** (axe Y) : un touriste motorisé peut physiquement se déplacer hors Paris

La **zone cible** (en haut à droite) = les marchés à fort taux de repeaters ET forte mobilité.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

top_markets = ["France", "Royaume-Uni", "Espagne", "Allemagne", "Italie",
               "Belgique", "Pays-Bas", "Suisse", "Etats-Unis", "Chine",
               "Japon", "Brésil", "Canada", "Australie", "Inde"]

df_pays_last = df_pays[(df_pays["annee"] == 2025) & (df_pays["pays"].isin(top_markets))].copy()
df_pays_last = df_pays_last.dropna(subset=["repeaters", "transport_route"])
df_pays_last["repeaters_pct"] = df_pays_last["repeaters"] * 100
df_pays_last["route_pct"] = df_pays_last["transport_route"] * 100
df_pays_last["sejours_m"] = df_pays_last["sejours"].fillna(1) / 1e6

scatter = ax.scatter(df_pays_last["repeaters_pct"], df_pays_last["route_pct"],
                     s=df_pays_last["sejours_m"] * 30, alpha=0.7,
                     c="#457B9D", edgecolors='white', linewidth=1.5)

for _, row in df_pays_last.iterrows():
    ax.annotate(row["pays"], (row["repeaters_pct"], row["route_pct"]),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

ax.axhline(y=30, color='#E63946', linestyle='--', alpha=0.5)
ax.axvline(x=70, color='#E63946', linestyle='--', alpha=0.5)
ax.text(85, 55, "ZONE CIBLE\n(repeaters + mobilité)", fontsize=10,
        color='#E63946', fontweight='bold', ha='center',
        bbox=dict(boxstyle='round', facecolor='#FFE5E5', alpha=0.8))

ax.set_xlabel("Taux de Repeaters (%)")
ax.set_ylabel("Part venus par la route (%)")
ax.set_title("Quels marchés sont les plus réorientables ?",
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig("graph5_marches_cibles.png", dpi=150)
plt.show()


### 📊 Graphique 6 — Potentiel économique de la dispersion

**Argument final** : Chaque jour supplémentaire passé en couronne génère des retombées économiques.

**Hypothèse** : si 10% des touristes de chaque département ajoutent 1 journée à leur séjour →
- Gain = dépense/jour × nombre de séjours × 10%


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

df_eco = df_dept[df_dept["annee"] == 2025][
    ["departement", "depense_jour_personne", "sejours", "duree_sejour"]
].dropna()
df_eco = df_eco[df_eco["departement"] != "Paris"]
df_eco["gain_1jour_millions"] = (df_eco["depense_jour_personne"] * df_eco["sejours"] * 0.10) / 1e6
df_eco = df_eco.sort_values("gain_1jour_millions", ascending=True)

colors = [COLORS.get(d, "#999") for d in df_eco["departement"]]
bars = ax.barh(df_eco["departement"], df_eco["gain_1jour_millions"], color=colors)

for bar, val, dep in zip(bars, df_eco["gain_1jour_millions"], df_eco["depense_jour_personne"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f} M€  (dép/j: {dep:.0f}€)', va='center', fontsize=10)

ax.set_xlabel("Gain estimé (millions €)")
ax.set_title("Potentiel : si 10% des touristes ajoutent 1 jour en couronne",
             fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig("graph6_potentiel_economique.png", dpi=150)
plt.show()


---

## 📋 Résumé — Chiffres clés pour le PowerPoint


In [ ]:
df_2025 = df_dept[df_dept["annee"] == 2025]
paris = df_2025[df_2025["departement"] == "Paris"].iloc[0]

print("=" * 60)
print("CHIFFRES CLÉS POUR LE POWERPOINT")
print("=" * 60)
print(f"""
🔴 CONSTAT — Paris craque
   • {paris['sejours_total_idf']/1e6:.1f}M séjours en IDF en 2025
   • Paris capte {paris['part_sejours_idf']:.1f}% des séjours
   • Satisfaction propreté Paris : {paris['satisfaction_proprete']*100:.1f}%
   • Satisfaction sécurité Paris : {paris['satisfaction_securite']*100:.1f}%
   • Paris perd {abs(paris.get('evol_sejours_pct', 0)):.1f}% de séjours en 1 an
""")

print("📈 OPPORTUNITÉ — La couronne explose")
for _, row in df_2025[df_2025["departement"] != "Paris"].sort_values("evol_sejours_pct", ascending=False).iterrows():
    evol = row.get("evol_sejours_pct", 0)
    if pd.notna(evol):
        emoji = "🟢" if evol > 0 else "🔴"
        print(f"   {emoji} {row['departement']}: {evol:+.1f}%")

print(f"""
🎯 CIBLE — Les repeaters motorisés
   • {paris['repeaters']*100:.1f}% de repeaters à Paris
   • Cible prioritaire : Européens proches (Belgique, Suisse, Allemagne)
     venus en voiture/train, séjour 5+ nuits, motif loisir

💰 POTENTIEL — Gain estimé si 10% ajoutent 1 jour en couronne
""")
for _, row in df_eco.sort_values("gain_1jour_millions", ascending=False).iterrows():
    print(f"   {row['departement']}: {row['gain_1jour_millions']:.0f} M€")
total_gain = df_eco["gain_1jour_millions"].sum()
print(f"   ─────────────────────────")
print(f"   TOTAL: {total_gain:.0f} M€")
